# ПЗ 7 — Распознавание объектов через Gemini Flash Lite

Используем самую дешёвую модель Gemini с паузами между запросами.

In [ ]:
!pip install google-generativeai pillow -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров всего: {len(frames)}')


In [ ]:
# читаем ключ из .env файла в Drive
env_path = '/content/drive/MyDrive/cv-frames/.env'

with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, _, val = line.partition('=')
            os.environ[key.strip()] = val.strip()

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
print(f'ключ загружен: {GEMINI_API_KEY[:8]}...')


In [ ]:
import json
import time
import google.generativeai as genai
from PIL import Image
from tqdm.notebook import tqdm

genai.configure(api_key=GEMINI_API_KEY)

# gemini-2.0-flash-lite — самая дешёвая модель с поддержкой изображений
gemini = genai.GenerativeModel('gemini-2.0-flash-lite')

# короткий промпт = меньше токенов = дешевле
PROMPT = 'Перечисли объекты на фото одной строкой через запятую. Только список, без пояснений.'

def describe_frame(image_path):
    img = Image.open(image_path)
    resp = gemini.generate_content([img, PROMPT])
    return {'objects': resp.text.strip(), 'frame': image_path}

# берём каждый 10-й кадр — не больше 5 штук
STEP       = max(1, len(frames) // 5)
BATCH      = frames[::STEP][:5]
PAUSE_SEC  = 10  # пауза между запросами

print(f'будем обрабатывать {len(BATCH)} кадров с паузой {PAUSE_SEC} сек')
print('кадры:', BATCH)


In [ ]:
results = []

for i, fname in enumerate(tqdm(BATCH, desc='gemini')):
    try:
        res = describe_frame(f'{FRAMES_DIR}/{fname}')
        results.append(res)
        print(f'{fname}: {res["objects"]}')
    except Exception as e:
        print(f'{fname}: ошибка — {e}')
        results.append({'frame': fname, 'objects': 'ошибка', 'error': str(e)})

    if i < len(BATCH) - 1:
        time.sleep(PAUSE_SEC)

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'\nобработано: {len(results)} кадров')
